In [ ]:
from json import JSONEncoder
import logging
import sys
import ndjson
from pickhardtpayments.pickhardtpayments import *

In [ ]:
def set_logger():
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    formatter = logging.Formatter('%(asctime)s.%(msecs)03d | %(levelname)s | %(message)s', datefmt='%H:%M:%S')
    stdout_handler = logging.StreamHandler(sys.stdout)
    stdout_handler.setLevel(logging.INFO)
    stdout_handler.setFormatter(formatter)
    file_handler = logging.FileHandler('pickhardt_pay.log', mode='w')
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)
    logger.addHandler(stdout_handler)

set_logger()
logging.info('*** new payment simulation ***')

In [ ]:
# subclass JSONEncoder
class PaymentEncoder(JSONEncoder):
    def default(self, o):
        return o.__dict__

In [ ]:
# *** Setup ***
# graph = ChannelGraph("./pickhardtpayments/channels.sample.json")
graph = ChannelGraph("./pickhardtpayments/listchannels20220412.json")

def only_channels_with_return_channels(_graph: ChannelGraph):
    channels_with_no_return_channel = []
    for edge in _graph.network.edges:
        if not _graph.network.has_edge(edge[1], edge[0]):
            channels_with_no_return_channel.append(edge)

    for edge in channels_with_no_return_channel:
        _graph.network.remove_edge(edge[0], edge[1], edge[2])

    if len(channels_with_no_return_channel) == 0:
        logging.debug("channel graph only had channels in both directions.")
    else:
        logging.debug("channel graph had unannounced channels.")
    return _graph

graph = only_channels_with_return_channels(graph)
uncertainty_network = UncertaintyNetwork(graph)
oracle_lightning_network = OracleLightningNetwork(graph)

In [ ]:
# test of liquidity adjustment
set_liquidity = 100000
import random
chan = random.sample(list(oracle_lightning_network.network.edges), min(1000, len(oracle_lightning_network.network.edges)))
liquidity_test = True
for chan in random.sample(list(oracle_lightning_network.network.edges), 5):
    c = oracle_lightning_network.get_channel(chan[0], chan[1], chan[2])
    if c.actual_liquidity != set_liquidity:
        liquidity_test = False
        print(f"liquidity not {set_liquidity:,}")
if liquidity_test:
    print(f"liquidity in sample is {set_liquidity:,}")

In [ ]:
sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
print("Setup finished")

* sample payment pairs
* sample success rate for payment to find out how large a sample should be to be stable
* sample success rate for pickhardt payment to find out how large a sample should be to be stable
* Decide on sample size
* save successful payments/Attempts from sample to json
* iterate over edges and find out "most frequent edges" in payment delivery